# Jewelry Eval Notebook

Run the eval pipeline from the modules in this repo and regenerate `eval/results/predictions.csv`, confusion matrices, and `misclassified.csv`.

This version is intended for Colab/Drive. It reads your metadata CSV with headers `original_filename`, `category`, `material`, and `labelled_path`, then samples 2,000 images in a balanced way across all available categories.

In [ ]:
from pathlib import Path, PureWindowsPath
import os

MOUNT_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    drive.mount(str(DRIVE_MOUNT_POINT))

# If running from the repo root, leave this as-is. In Colab, set it to the cloned repo path.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config" / "config.yaml").exists() and (PROJECT_ROOT.parent / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# CHANGE THESE TWO PATHS.
# METADATA_CSV must have: original_filename, category, material, labelled_path
METADATA_CSV = DRIVE_MOUNT_POINT / "MyDrive" / "rebuilt_labelled_metadata.csv"

# Folder where images are stored. Relative labelled_path values are resolved under this folder.
IMAGES_ROOT = DRIVE_MOUNT_POINT / "MyDrive" / "labelled_data" / "labelled_data"

CONFIG_YAML = PROJECT_ROOT / "config" / "config.yaml"
PROMPTS_YAML = PROJECT_ROOT / "config" / "prompts.yaml"
OUTPUT_DIR = PROJECT_ROOT / "eval" / "results"
SAMPLED_EVAL_CSV = OUTPUT_DIR / "balanced_eval_sample.csv"

BALANCED_SAMPLE_SIZE = 2000
RANDOM_STATE = 42

# Current repo preprocessing expects images that already have alpha/background removal.
# Turn this on to mimic the older Colab-style eval: RGB image -> BiRefNet RGBA -> ImageProcessor.
USE_SERVER_SIDE_BIREFNET = True

# Use a small integer for a smoke test, or None for the full eval set.
LIMIT = None

PROJECT_ROOT, METADATA_CSV, IMAGES_ROOT, OUTPUT_DIR

## Optional Colab Setup

Run this only in a fresh Colab runtime after cloning/uploading the repo and making model folders available at the paths in `config/config.yaml`.

In [ ]:
# Uncomment in Colab if dependencies are missing.
# %pip install -q -r requirements.txt seaborn scikit-learn matplotlib ipywidgets

# If your repo is not already the working directory in Colab, either cd into it first:
# %cd /content/jewelry-search-prod
# or set PROJECT_ROOT manually in the first cell.

In [ ]:
import os
import sys
import types
import yaml
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.insert(0, str(PROJECT_ROOT))

from engines.clip_engine import CLIPEngine
from engines.dinov2_engine import DINOv2Engine
from engines.birefnet_engine import BiRefNetEngine
from preprocess.image_processor import ImageProcessor
from classifiers.category_classifier import CategoryClassifier
from classifiers.material_classifier import MaterialClassifier
from indexing.pipeline import IndexingPipeline

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Writing results to: {OUTPUT_DIR}")

In [ ]:
def load_config(path: Path):
    with open(path, "r") as f:
        raw = yaml.safe_load(f)
    return types.SimpleNamespace(**raw.get("inference", {}))


def load_prompts(path: Path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


config = load_config(CONFIG_YAML)
prompts = load_prompts(PROMPTS_YAML)

# Optional Colab path overrides. Uncomment and edit if model folders live elsewhere.
# config.birefnet_model_path = "/content/drive/MyDrive/models/BiRefNet"
# config.clip_model_path = "/content/drive/MyDrive/models/clip-ViT-L-14"

print(config)
print("Categories:", list(prompts["categories"].keys()))
print("Materials:", list(prompts["materials"].keys()))

In [ ]:
def resolve_image_path(labelled_path_value, original_filename_value=None, category_value=None) -> Path:
    raw = "" if pd.isna(labelled_path_value) else str(labelled_path_value).strip()
    filename = PureWindowsPath(raw).name if "\\" in raw else Path(raw).name
    path = Path(raw.replace("\\", "/"))

    candidates = []
    if path.is_absolute():
        candidates.append(path)
    else:
        candidates.extend([PROJECT_ROOT / path, IMAGES_ROOT / path])

    category = None if category_value is None or pd.isna(category_value) else str(category_value).strip()
    if category:
        candidates.extend([IMAGES_ROOT / category / path, IMAGES_ROOT / category / filename])

    if raw:
        candidates.append(IMAGES_ROOT / filename)

    if original_filename_value is not None and not pd.isna(original_filename_value):
        original_filename = str(original_filename_value).strip()
        candidates.append(IMAGES_ROOT / original_filename)
        if category:
            candidates.append(IMAGES_ROOT / category / original_filename)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return candidates[0] if candidates else IMAGES_ROOT / raw


def balanced_sample_by_category(df: pd.DataFrame, total: int, random_state: int) -> pd.DataFrame:
    categories = sorted(df["category"].dropna().unique())
    if not categories:
        raise ValueError("No categories found after cleaning metadata.")

    shuffled_groups = {
        category: group.sample(frac=1, random_state=random_state)
        for category, group in df.groupby("category", sort=True)
    }
    positions = {category: 0 for category in categories}
    selected_indices = []

    while len(selected_indices) < total:
        progressed = False
        for category in categories:
            group = shuffled_groups[category]
            pos = positions[category]
            if pos < len(group):
                selected_indices.append(group.index[pos])
                positions[category] += 1
                progressed = True
                if len(selected_indices) == total:
                    break
        if not progressed:
            break

    return df.loc[selected_indices].sample(frac=1, random_state=random_state).reset_index(drop=True)


metadata = pd.read_csv(METADATA_CSV)
required = {"original_filename", "category", "material", "labelled_path"}
missing_cols = required - set(metadata.columns)
if missing_cols:
    raise ValueError(f"Metadata CSV is missing columns: {sorted(missing_cols)}")

metadata = metadata.copy()
metadata["category"] = metadata["category"].astype("string").str.strip()
metadata = metadata[metadata["category"].notna() & (metadata["category"] != "")]
metadata["material"] = metadata["material"].fillna("unknown").astype("string").str.strip()
metadata.loc[metadata["material"] == "", "material"] = "unknown"

metadata["image_path"] = metadata.apply(
    lambda row: str(resolve_image_path(row["labelled_path"], row["original_filename"], row["category"])),
    axis=1,
)
metadata["filename"] = metadata["image_path"].apply(lambda value: Path(value).name)
metadata["serial_number"] = metadata["original_filename"].apply(lambda value: PureWindowsPath(str(value)).stem)
duplicate_serials = metadata["serial_number"].duplicated(keep=False)
metadata.loc[duplicate_serials, "serial_number"] = (
    metadata.loc[duplicate_serials, "serial_number"].astype(str) + "__row_" + metadata.loc[duplicate_serials].index.astype(str)
)

print(f"Metadata rows after cleaning: {len(metadata)}")
display(metadata["category"].value_counts().rename("available_count").to_frame())

gt = balanced_sample_by_category(metadata, BALANCED_SAMPLE_SIZE, RANDOM_STATE)
if LIMIT is not None:
    gt = gt.head(LIMIT).copy()

gt.to_csv(SAMPLED_EVAL_CSV, index=False)
print(f"Balanced eval rows: {len(gt)}")
print(f"Saved sampled eval CSV: {SAMPLED_EVAL_CSV}")
display(gt.head())
display(gt["category"].value_counts().rename("sampled_count").to_frame())

In [ ]:
print("Loading engines...")
birefnet = BiRefNetEngine(config) if USE_SERVER_SIDE_BIREFNET else None
clip = CLIPEngine(config)
dinov2 = DINOv2Engine(config)

print("Loading classifiers...")
cat_clf = CategoryClassifier(clip, prompts["categories"])
mat_clf = MaterialClassifier(clip, prompts["materials"])

# Do not pass BiRefNet into ImageProcessor in the current repo version.
processor = ImageProcessor()
pipeline = IndexingPipeline(processor, clip, dinov2, cat_clf, mat_clf)
print("Pipeline ready.")

In [ ]:
def image_path_for(row) -> Path:
    return Path(str(row["image_path"]))


missing = []
for _, row in gt.iterrows():
    path = image_path_for(row)
    if not path.exists():
        missing.append(str(path))

print(f"Missing images: {len(missing)}")
if missing:
    print("First missing paths:")
    for path in missing[:10]:
        print("  ", path)

In [ ]:
predictions = []

for i, row in gt.reset_index(drop=True).iterrows():
    img_path = image_path_for(row)
    if not img_path.exists():
        print(f"MISSING: {img_path}")
        continue

    try:
        opened_image = Image.open(img_path)
        if USE_SERVER_SIDE_BIREFNET:
            image_for_pipeline = birefnet.get_rgba(opened_image.convert("RGB"))
        else:
            image_for_pipeline = opened_image.convert("RGBA")

        record = pipeline.process(
            serial_number=str(row["serial_number"]),
            image=image_for_pipeline,
            tenant_id="eval",
        )
        predictions.append({
            "serial_number": row["serial_number"],
            "predicted_category": record.category,
            "category_confidence": record.category_confidence,
            "predicted_material": record.material,
            "material_confidence": record.material_confidence,
            "error": record.error,
        })
    except Exception as exc:
        print(f"ERROR {row['serial_number']}: {exc}")

    if (i + 1) % 10 == 0 or (i + 1) == len(gt):
        print(f"Processed {i + 1}/{len(gt)}")

preds_df = pd.DataFrame(predictions)
predictions_path = OUTPUT_DIR / "predictions.csv"
preds_df.to_csv(predictions_path, index=False)
print(f"Saved: {predictions_path}")
display(preds_df.head())

In [ ]:
df = gt.merge(preds_df, on="serial_number")
df = df[df["error"].isna()].copy()
print(f"Successfully processed: {len(df)}/{len(gt)}")

print("\n=== CATEGORY HEAD ===")
cat_acc = accuracy_score(df["category"], df["predicted_category"])
print(f"Accuracy: {cat_acc:.1%}")
print(classification_report(df["category"], df["predicted_category"], zero_division=0))

mat_df = df[df["material"] != "unknown"].copy()
if len(mat_df) > 0:
    print("\n=== MATERIAL HEAD ===")
    mat_acc = accuracy_score(mat_df["material"], mat_df["predicted_material"])
    print(f"Accuracy: {mat_acc:.1%}")
    print(classification_report(mat_df["material"], mat_df["predicted_material"], zero_division=0))
else:
    mat_acc = None
    print("\nNo non-unknown material labels found; skipping material metrics.")

In [ ]:
def save_confusion_matrix(y_true, y_pred, labels, title, cmap, path: Path, figsize):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap=cmap)
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"Saved: {path}")


cat_labels = sorted(df["category"].unique())
save_confusion_matrix(
    df["category"],
    df["predicted_category"],
    cat_labels,
    f"Category Confusion Matrix (Accuracy: {cat_acc:.1%})",
    "Reds",
    OUTPUT_DIR / "confusion_matrix_category.png",
    (14, 10),
)

if len(mat_df) > 0:
    mat_labels = sorted(mat_df["material"].unique())
    save_confusion_matrix(
        mat_df["material"],
        mat_df["predicted_material"],
        mat_labels,
        f"Material Confusion Matrix (Accuracy: {mat_acc:.1%})",
        "Blues",
        OUTPUT_DIR / "confusion_matrix_material.png",
        (10, 8),
    )

In [ ]:
misclassified = df[
    (df["category"] != df["predicted_category"]) |
    (df["material"] != df["predicted_material"])
][[
    "serial_number",
    "category",
    "predicted_category",
    "category_confidence",
    "material",
    "predicted_material",
    "material_confidence",
]]

misclassified_path = OUTPUT_DIR / "misclassified.csv"
misclassified.to_csv(misclassified_path, index=False)
print(f"Misclassified: {len(misclassified)}/{len(df)}")
print(f"Saved: {misclassified_path}")
display(misclassified.head(20))

## Notes

- `eval/run_eval.py` has stale preprocessing wiring for the current repo: it calls `ImageProcessor(birefnet)`, but `ImageProcessor` now expects no engine and accepts RGBA input.
- This notebook reads your metadata CSV directly, creates a balanced 2,000-image eval sample, and saves that sample to `eval/results/balanced_eval_sample.csv`.
- It keeps the current module boundary and optionally runs `BiRefNetEngine.get_rgba(...)` before `pipeline.process(...)` so older RGB eval images can still follow the same intended path.
- If DINOv2 is not cached, `torch.hub.load("facebookresearch/dinov2", ...)` needs network access the first time.